# Camera Calibration & Homography v5
> 클릭 좌표 수집: Canvas 재렌더링 버그 수정 / 한글 폰트 안정화


## Step 0 — 설치 및 임포트

In [ ]:
!pip install opencv-python-headless Pillow -q
!apt-get install -y fonts-nanum -qq > /dev/null 2>&1

import cv2, numpy as np, json, os, base64, io
from IPython.display import display, HTML
import matplotlib, matplotlib.pyplot as plt, matplotlib.font_manager as fm
from google.colab import output as colab_output

# 한글 폰트
nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(nanum):
    fe = fm.FontEntry(fname=nanum, name='NanumGothic')
    fm.fontManager.ttflist.insert(0, fe)
    matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False
print("완료")


## Step 1 — 이미지 업로드

In [ ]:
from google.colab import files
print("공장 카메라 이미지를 업로드하세요 (jpg/png)")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
img_bgr = cv2.imread(filename)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
H_px, W_px = img_bgr.shape[:2]
print(f"업로드 완료: {filename}  ({W_px} x {H_px} px)")


## Step 2 — 이미지 클릭으로 기준점 수집

셀 실행 후 이미지가 표시됩니다.  
**기준점 위치를 클릭** → 빨간 점 + 번호가 찍히고 좌표가 출력됩니다.  
잘못 찍었으면 **[되돌리기]**, 다 찍었으면 **[완료]** 버튼을 누르세요.

**좋은 기준점 위치**
- 기둥이 바닥에 닿는 하단 모서리
- 노란 안전선이 꺾이는 점
- 최소 6개, 화면 전체에 골고루


In [ ]:

from PIL import Image as PILImage

# ── 이미지 → base64 (한 번만 인코딩, 재사용) ──────────────
MAX_W = 900
scale = min(1.0, MAX_W / W_px)
disp_w = int(W_px * scale)
disp_h = int(H_px * scale)

pil_disp = PILImage.fromarray(img_rgb).resize((disp_w, disp_h), PILImage.LANCZOS)
buf = io.BytesIO()
pil_disp.save(buf, format='PNG')
IMG_B64 = base64.b64encode(buf.getvalue()).decode()

clicked_pts = []   # 수집된 좌표 저장

# ── Python 콜백: JS에서 좌표를 받아 처리 ─────────────────
def on_canvas_click(cx, cy):
    rx = int(round(cx / scale))
    ry = int(round(cy / scale))
    clicked_pts.append([rx, ry])
    n = len(clicked_pts)
    # JS에 점 추가 명령 전달 (이미지 재로드 없이 점만 추가)
    display(HTML(f"""
<script>
addDot({cx}, {cy}, {n});
</script>
"""))
    print(f"  {n}번 클릭 → 픽셀 ({rx}, {ry})")

def on_undo_click(dummy=None):
    if not clicked_pts:
        print("삭제할 점이 없습니다.")
        return
    p = clicked_pts.pop()
    print(f"  삭제: {p}  |  남은 점: {len(clicked_pts)}개")
    # 전체 점 목록을 JS로 전달해 캔버스 재그림
    dots_js = json.dumps([[int(p[0]*scale), int(p[1]*scale)] for p in clicked_pts])
    display(HTML(f"""
<script>
redrawAll({dots_js});
</script>
"""))

def on_done_click(dummy=None):
    print()
    print(f"=== 완료: {len(clicked_pts)}개 기준점 확정 ===")
    print()
    print("pixel_pts = np.array([")
    for i, p in enumerate(clicked_pts):
        print(f"    [{p[0]}, {p[1]}],   # {i+1}번")
    print("], dtype=np.float32)")

colab_output.register_callback('nb.click',  on_canvas_click)
colab_output.register_callback('nb.undo',   on_undo_click)
colab_output.register_callback('nb.done',   on_done_click)

# ── HTML Canvas: 이미지를 JS 변수에 저장해 재사용 ─────────
display(HTML(f"""
<div style="font-family:sans-serif;margin-bottom:8px;">
  <button onclick="google.colab.kernel.invokeFunction('nb.undo',[0],{{}})"
          style="background:#f0ad4e;border:none;padding:6px 14px;border-radius:4px;
                 cursor:pointer;font-size:13px;margin-right:8px;">되돌리기</button>
  <button onclick="google.colab.kernel.invokeFunction('nb.done',[0],{{}})"
          style="background:#5cb85c;color:white;border:none;padding:6px 14px;
                 border-radius:4px;cursor:pointer;font-size:13px;">완료</button>
  <span id="dot_count" style="margin-left:12px;font-size:13px;color:#555;">
    클릭한 점: 0개</span>
</div>

<canvas id="cvs" width="{disp_w}" height="{disp_h}"
        style="border:2px solid #555;cursor:crosshair;display:block;"></canvas>

<script>
// 이미지를 JS 변수에 저장 (한 번만 로드)
var _img = new Image();
var _pts = [];   // 점 목록 (표시 좌표)

function drawCanvas() {{
  var c = document.getElementById('cvs');
  if (!c) return;
  var ctx = c.getContext('2d');
  ctx.clearRect(0, 0, c.width, c.height);
  ctx.drawImage(_img, 0, 0);          // 이미지 먼저
  for (var i = 0; i < _pts.length; i++) {{
    var x = _pts[i][0], y = _pts[i][1];
    ctx.beginPath(); ctx.arc(x, y, 9, 0, 2*Math.PI);
    ctx.fillStyle = 'rgba(255,40,40,0.9)'; ctx.fill();
    ctx.strokeStyle = 'white'; ctx.lineWidth = 2; ctx.stroke();
    ctx.fillStyle = 'white'; ctx.font = 'bold 14px sans-serif';
    ctx.fillText(i+1, x+12, y-4);
  }}
  document.getElementById('dot_count').innerText =
    '클릭한 점: ' + _pts.length + '개';
}}

// 점 추가 (이미지 재로드 없이)
function addDot(cx, cy, n) {{
  _pts.push([cx, cy]);
  drawCanvas();
}}

// 전체 재그림 (되돌리기용)
function redrawAll(pts) {{
  _pts = pts;
  drawCanvas();
}}

// 클릭 이벤트
document.getElementById('cvs').addEventListener('click', function(e) {{
  var r = this.getBoundingClientRect();
  var sx = this.width  / r.width;
  var sy = this.height / r.height;
  var cx = Math.round((e.clientX - r.left) * sx);
  var cy = Math.round((e.clientY - r.top)  * sy);
  google.colab.kernel.invokeFunction('nb.click', [cx, cy], {{}});
}});

// 이미지 로드 후 초기 그리기
_img.onload = function() {{ drawCanvas(); }};
_img.src = 'data:image/png;base64,{IMG_B64}';
</script>
"""))


In [ ]:
# 완료 버튼 누른 후 이 셀 실행 — pixel_pts 변수 생성
pixel_pts = np.array(clicked_pts, dtype=np.float32)
print(f"기준점 {len(pixel_pts)}개 확정")
for i, p in enumerate(pixel_pts):
    print(f"  {i+1}번: ({int(p[0])}, {int(p[1])})")

if len(pixel_pts) < 4:
    print("\n경고: 최소 4개 이상 필요합니다. Step 2를 다시 실행하세요.")


## Step 3 — 실측 좌표 입력

Step 2 클릭 순서와 **동일한 순서**로 줄자로 잰 실제 거리를 입력합니다.

- **1번 기준점 = 원점 (0, 0)**
- X = 좌우(가로 방향), Y = 앞뒤(깊이 방향)
- 단위: 미터


In [ ]:
# ↓ 줄자로 잰 실측값으로 채우세요 (단위: 미터)
real_pts = np.array([
    [ 0.0,  0.0],   # 1번 원점
    [ 6.0,  1.0],   # 2번 — 오른쪽으로 6m, 약간 더 멀리
    [-6.0,  0.5],   # 3번 — 왼쪽으로 6m, 비슷한 깊이
    [-3.5,  2.5],   # 4번 — 왼쪽 3.5m, 앞쪽 2.5m 더 멀리
    [ 5.5, -3.5],   # 5번 — 오른쪽 5.5m, 카메라 방향으로 3.5m 가까움
    [ 1.5,  3.5],   # 6번 — 약간 오른쪽, 3.5m 더 멀리
], dtype=np.float32)

assert len(pixel_pts) == len(real_pts), \
    f"개수 불일치: 픽셀 {len(pixel_pts)}개 / 실측 {len(real_pts)}개"

print(f"{'No':<4} {'픽셀 (u,v)':<22} {'실측 (X,Y) m'}")
print("-" * 44)
for i, (p, r) in enumerate(zip(pixel_pts, real_pts)):
    print(f"{i+1:<4} ({int(p[0]):>5}, {int(p[1]):>5})          ({r[0]:.2f}, {r[1]:.2f})")


## Step 4 — H 행렬 계산

In [ ]:
H_mat, mask = cv2.findHomography(pixel_pts, real_pts, cv2.RANSAC, 5.0)

print("H matrix (3x3):")
for row in H_mat:
    print("  " + "  ".join(f"{v:13.7f}" for v in row))
inliers = int(mask.sum())
print(f"\nInliers: {inliers} / {len(pixel_pts)}")
print("OK" if inliers >= 4 else "WARNING: 기준점 재확인 필요")

with open("homography.json", "w") as f:
    json.dump({"H": H_mat.tolist(),
               "pixel_pts": pixel_pts.tolist(),
               "real_pts":  real_pts.tolist()}, f, indent=2)
print("homography.json 저장 완료")


## Step 5 — 발 좌표 기준 실거리 계산

바운딩박스 **하단 중심(발/바퀴)** 을 H에 입력해야 정확합니다.

In [ ]:
# Step 5 — 드래그로 바운딩박스 그리기 + 실거리 계산 (통합 버전)

from PIL import Image as PILImage
from google.colab import output as colab_output
from IPython.display import display, HTML
from matplotlib.patches import Rectangle
import json, base64, io

bboxes = []
bbox_labels = []
COLORS = ['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6','#1abc9c']

MAX_W = 900
scale5 = min(1.0, MAX_W / W_px)
dw5 = int(W_px * scale5)
dh5 = int(H_px * scale5)

pil5 = PILImage.fromarray(img_rgb).resize((dw5, dh5), PILImage.LANCZOS)
buf5 = io.BytesIO()
pil5.save(buf5, format='PNG')
IMG_B64_5 = base64.b64encode(buf5.getvalue()).decode()

def pixel_to_real(u, v, H):
    pt = H @ np.array([u, v, 1.0])
    return pt[0]/pt[2], pt[1]/pt[2]

def on_bbox_drawn(x1, y1, x2, y2):
    rx1=int(round(min(x1,x2)/scale5)); ry1=int(round(min(y1,y2)/scale5))
    rx2=int(round(max(x1,x2)/scale5)); ry2=int(round(max(y1,y2)/scale5))
    n = len(bboxes)
    label = chr(65+n)
    bboxes.append((rx1,ry1,rx2,ry2))
    bbox_labels.append(label)
    sx1,sy1=int(min(x1,x2)),int(min(y1,y2))
    sx2,sy2=int(max(x1,x2)),int(max(y1,y2))
    color = COLORS[n % len(COLORS)]
    display(HTML(f"<script>addBbox({sx1},{sy1},{sx2},{sy2},'{label}','{color}');</script>"))
    print(f"  {label}: 픽셀 ({rx1},{ry1}) ~ ({rx2},{ry2})")

def on_bbox_undo(dummy=None):
    if not bboxes:
        return
    bboxes.pop(); removed = bbox_labels.pop()
    print(f"  {removed} 삭제")
    boxes_js = json.dumps([
        [int(min(b[0],b[2])*scale5),int(min(b[1],b[3])*scale5),
         int(max(b[0],b[2])*scale5),int(max(b[1],b[3])*scale5),
         chr(65+i), COLORS[i%len(COLORS)]]
        for i,b in enumerate(bboxes)
    ])
    display(HTML(f"<script>redrawBboxes({boxes_js});</script>"))

def on_bbox_calc(dummy=None):
    if len(bboxes) < 2:
        print("박스를 2개 이상 그려주세요.")
        return
    print(f"\n=== 실거리 계산 결과 ({len(bboxes)}개 박스) ===\n")
    fig, ax = plt.subplots(figsize=(14,8))
    ax.imshow(img_rgb)
    foot_pts = []
    for i,(b,lbl) in enumerate(zip(bboxes,bbox_labels)):
        color = COLORS[i%len(COLORS)]
        x1,y1,x2,y2 = b
        ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,lw=2,ec=color,fc='none'))
        fu,fv = (x1+x2)/2, float(y2)
        ax.plot(fu,fv,'o',color=color,ms=9,mec='white',mew=1.5)
        ax.text(x1,y1-8,lbl,color=color,fontsize=12,fontweight='bold')
        X,Y = pixel_to_real(fu,fv,H_mat)
        foot_pts.append((fu,fv,X,Y))
        print(f"  {lbl}: 실제 위치 ({X:.2f}, {Y:.2f}) m")
    print()
    for i in range(len(bboxes)):
        for j in range(i+1,len(bboxes)):
            Xi,Yi = foot_pts[i][2],foot_pts[i][3]
            Xj,Yj = foot_pts[j][2],foot_pts[j][3]
            dist = np.sqrt((Xi-Xj)**2+(Yi-Yj)**2)
            la,lb = bbox_labels[i],bbox_labels[j]
            print(f"  {la} ↔ {lb} 실거리: {dist:.2f} m")
            fa=(int(foot_pts[i][0]),int(foot_pts[i][1]))
            fb=(int(foot_pts[j][0]),int(foot_pts[j][1]))
            ax.plot([fa[0],fb[0]],[fa[1],fb[1]],'w-',lw=1.5,alpha=0.7)
            mx,my=(fa[0]+fb[0])//2,(fa[1]+fb[1])//2
            ax.text(mx+4,my-6,f"{dist:.2f}m",color='white',fontsize=9,
                    fontweight='bold',
                    bbox=dict(boxstyle='round',fc='black',alpha=0.55))
    ax.set_title("바운딩박스 실거리 계산 결과",fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

colab_output.register_callback('nb.bbox',      on_bbox_drawn)
colab_output.register_callback('nb.bbox_undo', on_bbox_undo)
colab_output.register_callback('nb.bbox_calc', on_bbox_calc)

display(HTML(f"""
<div style="font-family:sans-serif;margin-bottom:8px;display:flex;align-items:center;gap:8px;">
  <span style="font-size:13px;color:#555;">드래그로 박스 그리기</span>
  <button onclick="google.colab.kernel.invokeFunction('nb.bbox_undo',[0],{{}})"
          style="background:#f0ad4e;border:none;padding:6px 14px;border-radius:4px;
                 cursor:pointer;font-size:13px;">되돌리기</button>
  <button onclick="google.colab.kernel.invokeFunction('nb.bbox_calc',[0],{{}})"
          style="background:#5cb85c;color:white;border:none;padding:6px 14px;
                 border-radius:4px;cursor:pointer;font-size:13px;">실거리 계산</button>
</div>
<canvas id="cvs5" width="{dw5}" height="{dh5}"
        style="border:2px solid #555;cursor:crosshair;display:block;"></canvas>
<script>
(function(){{
  var c=document.getElementById('cvs5');
  var ctx=c.getContext('2d');
  var _img=new Image();
  var _boxes=[];
  var _drag=false, _sx, _sy;

  function draw(){{
    ctx.clearRect(0,0,c.width,c.height);
    ctx.drawImage(_img,0,0);
    for(var i=0;i<_boxes.length;i++){{
      var b=_boxes[i];
      ctx.strokeStyle=b[4]; ctx.lineWidth=2.5; ctx.setLineDash([]);
      ctx.strokeRect(b[0],b[1],b[2]-b[0],b[3]-b[1]);
      ctx.fillStyle=b[4]; ctx.font='bold 14px sans-serif';
      ctx.fillText(b[5],b[0]+5,b[1]+17);
    }}
  }}
  window.addBbox=function(x1,y1,x2,y2,lbl,color){{
    _boxes.push([x1,y1,x2,y2,color,lbl]); draw();
  }};
  window.redrawBboxes=function(arr){{
    _boxes=[];
    for(var i=0;i<arr.length;i++)
      _boxes.push([arr[i][0],arr[i][1],arr[i][2],arr[i][3],arr[i][5],arr[i][4]]);
    draw();
  }};
  c.addEventListener('mousedown',function(e){{
    var r=c.getBoundingClientRect();
    _sx=Math.round((e.clientX-r.left)*(c.width/r.width));
    _sy=Math.round((e.clientY-r.top)*(c.height/r.height));
    _drag=true;
  }});
  c.addEventListener('mousemove',function(e){{
    if(!_drag) return;
    var r=c.getBoundingClientRect();
    var cx=Math.round((e.clientX-r.left)*(c.width/r.width));
    var cy=Math.round((e.clientY-r.top)*(c.height/r.height));
    draw();
    ctx.strokeStyle='rgba(255,255,255,0.85)'; ctx.lineWidth=1.5; ctx.setLineDash([5,3]);
    ctx.strokeRect(_sx,_sy,cx-_sx,cy-_sy); ctx.setLineDash([]);
  }});
  c.addEventListener('mouseup',function(e){{
    if(!_drag) return; _drag=false;
    var r=c.getBoundingClientRect();
    var ex=Math.round((e.clientX-r.left)*(c.width/r.width));
    var ey=Math.round((e.clientY-r.top)*(c.height/r.height));
    if(Math.abs(ex-_sx)<5||Math.abs(ey-_sy)<5) return;
    google.colab.kernel.invokeFunction('nb.bbox',[_sx,_sy,ex,ey],{{}});
  }});
  _img.onload=function(){{ draw(); }};
  _img.src='data:image/png;base64,{IMG_B64_5}';
}})();
</script>
"""))

## Step 6 — Bird's-Eye View 조감도

In [ ]:
PPM = 60
xs, ys = real_pts[:,0], real_pts[:,1]
X_min, X_max = xs.min()-1, xs.max()+1
Y_min, Y_max = ys.min()-1, ys.max()+1
out_W = int((X_max-X_min)*PPM)
out_H = int((Y_max-Y_min)*PPM)

dst = np.array([[(r[0]-X_min)*PPM, (Y_max-r[1])*PPM]
                for r in real_pts], dtype=np.float32)
H_bev, _ = cv2.findHomography(pixel_pts, dst)
bev = cv2.warpPerspective(img_bgr, H_bev, (out_W, out_H))
bev_rgb = cv2.cvtColor(bev, cv2.COLOR_BGR2RGB)
bev_d = bev_rgb.copy()

for x_m in np.arange(X_min, X_max+1, 1.0):
    px = int((x_m-X_min)*PPM)
    if 0<=px<out_W: cv2.line(bev_d,(px,0),(px,out_H),(160,160,160),1)
for y_m in np.arange(Y_min, Y_max+1, 1.0):
    py = int((Y_max-y_m)*PPM)
    if 0<=py<out_H: cv2.line(bev_d,(0,py),(out_W,py),(160,160,160),1)
for i, r in enumerate(real_pts):
    bx = int((r[0]-X_min)*PPM)
    by = int((Y_max-r[1])*PPM)
    cv2.circle(bev_d,(bx,by),9,(0,220,0),-1)
    cv2.putText(bev_d,str(i+1),(bx+10,by+5),
                cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,200,0),2)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(img_rgb);  axes[0].set_title("원본 카메라 뷰"); axes[0].axis("off")
axes[1].imshow(bev_d);    axes[1].set_title("Bird's-Eye View"); axes[1].axis("off")
plt.suptitle("Homography 변환 결과", fontsize=14)
plt.tight_layout()
plt.show()

cv2.imwrite("bev_result.jpg", bev)
print("bev_result.jpg 저장 완료")


## 결과 다운로드

In [ ]:
from google.colab import files
for f in ["homography.json", "bev_result.jpg"]:
    if os.path.exists(f):
        files.download(f)
        print(f"다운로드: {f}")
